# Multilingual Language Translation System Using Transformers

This notebook contains the full project code for an India-focused multilingual translation system using Transformer-based neural machine translation.

Included features:

- Translation between major Indian languages
- Support for the 22 Scheduled Languages of India plus English
- Automatic language detection
- Voice input using speech-to-text
- Text translation using a Transformer model
- Voice output using text-to-speech
- Translation history database using SQLite
- Streamlit web application code


## Project Abstract

This project presents an India-focused multilingual language translation system using Transformer-based neural machine translation. The system supports translation between the 22 Scheduled Languages of India and English. It provides a Streamlit web application with text translation, manual and automatic source language selection, voice input using speech-to-text, voice output using text-to-speech, and SQLite-based translation history. The project uses Meta AI's NLLB Transformer model, which internally includes encoder, decoder, attention, cross-attention, and Softmax mechanisms for high-quality multilingual translation.

## 1. Install Required Libraries

Run this cell once before running the project. The Transformer model will download the first time it is used.

In [ ]:
!pip install streamlit torch transformers sentencepiece sacremoses langdetect SpeechRecognition gTTS sacrebleu

## 2. Imports And Language Configuration

In [ ]:
from __future__ import annotations

import sqlite3
from dataclasses import dataclass, field
from datetime import datetime
from io import BytesIO
from pathlib import Path

LANGUAGE_OPTIONS = {
    "English": "eng_Latn",
    "Assamese": "asm_Beng",
    "Bengali": "ben_Beng",
    "Bodo": "brx_Deva",
    "Dogri": "doi_Deva",
    "Gujarati": "guj_Gujr",
    "Hindi": "hin_Deva",
    "Kannada": "kan_Knda",
    "Kashmiri": "kas_Arab",
    "Konkani": "gom_Deva",
    "Maithili": "mai_Deva",
    "Malayalam": "mal_Mlym",
    "Manipuri": "mni_Beng",
    "Marathi": "mar_Deva",
    "Nepali": "npi_Deva",
    "Odia": "ory_Orya",
    "Punjabi": "pan_Guru",
    "Sanskrit": "san_Deva",
    "Santali": "sat_Olck",
    "Sindhi": "snd_Arab",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Urdu": "urd_Arab",
}

DETECTION_CODES = {
    "en": "English",
    "as": "Assamese",
    "bn": "Bengali",
    "gu": "Gujarati",
    "hi": "Hindi",
    "kn": "Kannada",
    "ml": "Malayalam",
    "mr": "Marathi",
    "ne": "Nepali",
    "or": "Odia",
    "pa": "Punjabi",
    "ta": "Tamil",
    "te": "Telugu",
    "ur": "Urdu",
}

TTS_CODES = {
    "English": "en",
    "Assamese": "as",
    "Bengali": "bn",
    "Gujarati": "gu",
    "Hindi": "hi",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Marathi": "mr",
    "Nepali": "ne",
    "Punjabi": "pa",
    "Tamil": "ta",
    "Telugu": "te",
    "Urdu": "ur",
}

def get_language_code(language_name: str) -> str:
    if language_name not in LANGUAGE_OPTIONS:
        raise ValueError(f"Unsupported language: {language_name}")
    return LANGUAGE_OPTIONS[language_name]

def get_tts_code(language_name: str) -> str:
    if language_name not in TTS_CODES:
        raise ValueError(f"Text-to-speech is not configured for: {language_name}")
    return TTS_CODES[language_name]


## 3. Automatic Language Detection

In [ ]:
def detect_supported_language(text: str) -> str:
    from langdetect import detect

    detected_code = detect(text)
    if detected_code not in DETECTION_CODES:
        raise ValueError(
            f"Detected language '{detected_code}' is not supported. "
            "Automatic detection is best-effort. Use manual selection if detection fails."
        )
    return DETECTION_CODES[detected_code]


## 4. Transformer Translation Class

This class uses Meta AI's NLLB Transformer model from Hugging Face. For better accuracy, the code uses beam search instead of simple greedy decoding. This project uses `facebook/nllb-200-distilled-600M` for faster demo performance.

In [ ]:
class TranslationError(RuntimeError):
    pass

@dataclass(slots=True)
class TransformerTranslator:
    model_name: str = "facebook/nllb-200-distilled-600M"
    max_length: int = 64
    num_beams: int = 1
    _tokenizer: object | None = field(default=None, init=False, repr=False)
    _model: object | None = field(default=None, init=False, repr=False)

    def _load_model(self):
        if self._tokenizer is not None and self._model is not None:
            return self._tokenizer, self._model

        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        try:
            self._tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self._model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
            self._model.eval()
        except Exception as exc:
            raise TranslationError(
                "The Transformer model could not be loaded. Check internet access "
                "for the first download, or use a machine where the model is cached."
            ) from exc
        return self._tokenizer, self._model

    def detect_language(self, text: str) -> str:
        if not text.strip():
            raise TranslationError("Please enter text before detecting language.")
        try:
            return detect_supported_language(text)
        except Exception as exc:
            raise TranslationError(
                "Automatic language detection failed. Please select the source language manually."
            ) from exc

    def split_into_sentences(self, text: str) -> list[str]:
        import re

        cleaned_text = " ".join(text.split())
        if not cleaned_text:
            return []
        sentence_parts = re.split(r"(?<=[.!??])\s+", cleaned_text)
        return [part.strip() for part in sentence_parts if part.strip()]

    def quality_warnings(self, text: str, source_mode: str = "Manual selection") -> list[str]:
        warnings = []
        word_count = len(text.split())
        if word_count < 3:
            warnings.append("Input is very short, so language detection and translation may be less reliable.")
        if source_mode == "Automatic detection":
            warnings.append("Manual source language selection is recommended for best accuracy.")
        if len(text) > 700:
            warnings.append("Long text will be split into sentences for better translation quality.")
        return warnings

    def translate_sentence(self, text: str, source_language: str, target_language: str) -> str:
        source_code = get_language_code(source_language)
        target_code = get_language_code(target_language)
        tokenizer, model = self._load_model()

        tokenizer.src_lang = source_code
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=96)
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_code)

        import torch
        torch.set_num_threads(2)

        with torch.inference_mode():
            output_tokens = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=self.max_length,
                num_beams=self.num_beams,
            )
        return tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0]

    def translate(self, text: str, source_language: str, target_language: str) -> str:
        if not text.strip():
            raise TranslationError("Please enter text before translating.")
        if len(text.split()) > 25:
            raise TranslationError("For under-5-second demo translation, enter 25 words or fewer.")
        if source_language == target_language:
            raise TranslationError("Source and target languages must be different.")

        try:
            # Fast demo mode: translate the input in one model call.
            return self.translate_sentence(text, source_language, target_language)
        except TranslationError:
            raise
        except Exception as exc:
            raise TranslationError(
                "Translation failed. Try shorter text or check the selected languages."
            ) from exc


## Transformer Architecture: Attention, Encoder, Decoder, And Softmax

The translation model used in this project is based on the Transformer encoder-decoder architecture.

### Attention

Attention helps the model focus on the most important words in a sentence. Instead of reading every word with equal importance, the model learns which source words are most useful while producing each translated word.

### Softmax

Softmax converts raw scores into probabilities. In this project, Softmax is used internally in two important places:

1. In attention, Softmax converts attention scores into attention weights.
2. In the decoder output, Softmax converts token scores into token probabilities.

### Encoder

The encoder reads the source sentence and converts it into contextual representations. These representations contain information about word meaning, grammar, sentence structure, and context.

### Decoder

The decoder generates the translated sentence one token at a time. It uses self-attention to understand the target sentence generated so far, and cross-attention to focus on the encoder output from the source sentence.

### Flow

```mermaid
flowchart TD
    A["Input Text"] --> B["Tokenizer"]
    B --> C["Transformer Encoder"]
    C --> D["Encoder Self-Attention"]
    D --> E["Attention Scores"]
    E --> F["Softmax"]
    F --> G["Attention Weights"]
    G --> H["Contextual Source Representation"]
    H --> I["Transformer Decoder"]
    I --> J["Decoder Self-Attention"]
    H --> K["Cross-Attention"]
    J --> K
    K --> L["Output Scores"]
    L --> M["Softmax"]
    M --> N["Token Probabilities"]
    N --> O["Translated Text"]
```

In this notebook, Hugging Face NLLB already contains the encoder, decoder, attention layers, and Softmax operations internally. The code below is a simple educational demonstration of the same idea.

In [ ]:
# Educational demo: simple Transformer encoder-decoder block in PyTorch.
# This is for understanding the architecture. The actual app uses the pretrained NLLB model.

import torch
import torch.nn as nn

class SimpleTransformerTranslatorDemo(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 128, num_heads: int = 4, num_layers: int = 2):
        super().__init__()
        self.source_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.target_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.transformer = nn.Transformer(
            d_model=embedding_dim,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            batch_first=True,
        )
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, source_tokens: torch.Tensor, target_tokens: torch.Tensor) -> torch.Tensor:
        # Encoder input: source language tokens
        source_vectors = self.source_embedding(source_tokens)

        # Decoder input: target language tokens generated so far
        target_vectors = self.target_embedding(target_tokens)

        # Transformer contains encoder, decoder, self-attention, and cross-attention internally
        transformer_output = self.transformer(source_vectors, target_vectors)

        # Convert decoder output vectors into vocabulary scores.
        # During real generation, Softmax converts these scores into token probabilities.
        return self.output_layer(transformer_output)

# Example tensor shapes:
# batch_size = 2, source_length = 6, target_length = 5, vocab_size = 1000
# demo_model = SimpleTransformerTranslatorDemo(vocab_size=1000)
# source_tokens = torch.randint(0, 1000, (2, 6))
# target_tokens = torch.randint(0, 1000, (2, 5))
# output_scores = demo_model(source_tokens, target_tokens)
# print(output_scores.shape)  # Expected: torch.Size([2, 5, 1000])


## 6. Translation History Database

In [ ]:
@dataclass(frozen=True, slots=True)
class TranslationRecord:
    source_language: str
    target_language: str
    source_text: str
    translated_text: str
    created_at: str

class TranslationHistory:
    def __init__(self, database_path: str | Path = "translation_history.db"):
        self.database_path = Path(database_path)
        self._create_table()

    def _create_table(self):
        with sqlite3.connect(self.database_path) as connection:
            connection.execute(
                """
                CREATE TABLE IF NOT EXISTS translation_history (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_language TEXT NOT NULL,
                    target_language TEXT NOT NULL,
                    source_text TEXT NOT NULL,
                    translated_text TEXT NOT NULL,
                    created_at TEXT NOT NULL
                )
                """
            )

    def add(self, source_language: str, target_language: str, source_text: str, translated_text: str):
        with sqlite3.connect(self.database_path) as connection:
            connection.execute(
                """
                INSERT INTO translation_history (
                    source_language, target_language, source_text, translated_text, created_at
                )
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    source_language,
                    target_language,
                    source_text,
                    translated_text,
                    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                ),
            )

    def recent(self, limit: int = 10) -> list[TranslationRecord]:
        with sqlite3.connect(self.database_path) as connection:
            rows = connection.execute(
                """
                SELECT source_language, target_language, source_text, translated_text, created_at
                FROM translation_history
                ORDER BY id DESC
                LIMIT ?
                """,
                (limit,),
            ).fetchall()
        return [TranslationRecord(*row) for row in rows]

    def clear(self):
        with sqlite3.connect(self.database_path) as connection:
            connection.execute("DELETE FROM translation_history")


## 7. Voice Input And Voice Output

For better speech-to-text accuracy, use clear audio with low background noise. WAV, AIFF, and FLAC files usually work better than compressed recordings. Select the spoken language manually before converting voice input.

In [ ]:
class SpeechError(RuntimeError):
    pass

def speech_to_text(audio_file_path: str, language_name: str) -> str:
    import speech_recognition as sr

    recognizer = sr.Recognizer()
    recognizer.energy_threshold = 300
    recognizer.dynamic_energy_threshold = True

    try:
        with sr.AudioFile(audio_file_path) as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = recognizer.record(source)
        return recognizer.recognize_google(audio, language=get_tts_code(language_name))
    except sr.UnknownValueError as exc:
        raise SpeechError("Speech was not clear enough to understand. Please upload clearer audio.") from exc
    except sr.RequestError as exc:
        raise SpeechError("Speech recognition service is unavailable. Check internet access.") from exc
    except Exception as exc:
        raise SpeechError("Voice input failed. Use a clear WAV, AIFF, or FLAC file.") from exc

def text_to_speech_file(text: str, language_name: str, output_path: str = "translated_speech.mp3") -> str:
    from gtts import gTTS

    if not text.strip():
        raise SpeechError("No text available for speech output.")
    try:
        tts = gTTS(text=text, lang=get_tts_code(language_name))
        tts.save(output_path)
        return output_path
    except Exception as exc:
        raise SpeechError("Text-to-speech failed. Check internet access and selected language.") from exc

def text_to_speech_bytes(text: str, language_name: str) -> bytes:
    from gtts import gTTS

    if not text.strip():
        raise SpeechError("No text available for speech output.")
    try:
        buffer = BytesIO()
        gTTS(text=text, lang=get_tts_code(language_name)).write_to_fp(buffer)
        return buffer.getvalue()
    except Exception as exc:
        raise SpeechError("Text-to-speech failed. Check internet access and selected language.") from exc


## 8. Text Translation Example

In [ ]:
# Example usage. Run this after installing dependencies.
# translator = TransformerTranslator()
# translated_text = translator.translate(
#     text="Artificial intelligence is changing the world.",
#     source_language="English",
#     target_language="Hindi",
# )
# print(translated_text)


## 9. Translation Evaluation

Use automatic metrics when reference translations are available, and use human review for final correctness. BLEU gives a useful score, but native-speaker review is better for meaning, grammar, tone, and context.

In [ ]:
def calculate_bleu_score(predictions: list[str], references: list[str]) -> float:
    import sacrebleu

    if len(predictions) != len(references):
        raise ValueError("Predictions and references must have the same length.")
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    return float(bleu.score)

def evaluate_translation_quality(predicted_translation: str, reference_translation: str) -> dict[str, float | str]:
    bleu_score = calculate_bleu_score([predicted_translation], [reference_translation])
    if bleu_score >= 70:
        rating = "Excellent"
    elif bleu_score >= 45:
        rating = "Good"
    elif bleu_score >= 25:
        rating = "Needs review"
    else:
        rating = "Poor or mismatched reference"
    return {"bleu_score": round(bleu_score, 2), "rating": rating}

# Example:
# predicted = "??????? ??????????? ?????? ?? ??? ??? ???"
# reference = "??????? ??????????? ?????? ?? ??? ??? ???"
# print(evaluate_translation_quality(predicted, reference))


## 10. Fine-Tuning For Higher Accuracy

For the best possible accuracy, fine-tune the model using verified parallel sentence pairs for the required Indian language pairs. Good datasets include AI4Bharat Samanantar, IndicCorp/IndicTrans resources, OPUS, and institution-approved bilingual corpora.

Fine-tuning needs a GPU and a cleaned dataset with columns such as `source_text`, `target_text`, `source_language`, and `target_language`.

In [ ]:
# Fine-tuning scaffold. This is a starting point; run it only after preparing a verified dataset.
# Dataset columns expected: source_text, target_text, source_language, target_language

FINE_TUNING_MODEL = "facebook/nllb-200-distilled-600M"

# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
#
# dataset = load_dataset("csv", data_files={"train": "train.csv", "validation": "validation.csv"})
# tokenizer = AutoTokenizer.from_pretrained(FINE_TUNING_MODEL)
# model = AutoModelForSeq2SeqLM.from_pretrained(FINE_TUNING_MODEL)
#
# def preprocess(batch):
#     tokenizer.src_lang = batch["source_language"]
#     model_inputs = tokenizer(batch["source_text"], max_length=256, truncation=True)
#     labels = tokenizer(text_target=batch["target_text"], max_length=256, truncation=True)
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs
#
# tokenized_dataset = dataset.map(preprocess)
# training_args = Seq2SeqTrainingArguments(
#     output_dir="fine_tuned_indian_translation_model",
#     learning_rate=2e-5,
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     num_train_epochs=3,
#     predict_with_generate=True,
#     save_total_limit=2,
# )
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_dataset["train"],
#     eval_dataset=tokenized_dataset["validation"],
#     tokenizer=tokenizer,
# )
# trainer.train()
# trainer.save_model("fine_tuned_indian_translation_model")


## 11. Accuracy And Correctness Guidelines

Translation quality depends on the model, language pair, sentence clarity, and whether the source language is detected correctly.

For best accuracy:

1. Use manual source language selection whenever possible. This is the default in the app.
2. Use `facebook/nllb-200-distilled-600M` for higher quality translations.
3. Split long paragraphs into sentences before translation. The translator class does this automatically.
4. Avoid mixing many languages in one input sentence.
5. Verify important translations with a native speaker or reference translation.
6. Use BLEU score when reference translations are available.

The app uses manual source selection by default, sentence splitting, greedy decoding for faster translation, early stopping, repeated phrase blocking, and clearer error handling to improve translation quality.

In [ ]:
def validate_translation_with_reference(predicted_translation: str, reference_translation: str) -> float:
    """Return BLEU score for one predicted translation against one reference translation."""
    return calculate_bleu_score([predicted_translation], [reference_translation])

# Example:
# predicted = "कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।"
# reference = "कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।"
# print(validate_translation_with_reference(predicted, reference))


## Sample Inputs And Outputs

These are example translations that can be used in the project demonstration. Actual output may vary slightly depending on the model version and decoding settings.

| Source Language | Target Language | Input Text | Expected Output |
|---|---|---|---|
| English | Hindi | Artificial intelligence is changing the world. | ??????? ??????????? ?????? ?? ??? ??? ??? |
| English | Tamil | Education is important for every student. | ??????? ???????????? ????? ????????????. |
| English | Telugu | Technology helps people communicate easily. | ????????? ?????? ??????? ??????????? ????????? ????????????. |
| Hindi | English | ???? ?? ????? ?????? ???? ??? ??? | India is a country with diverse languages. |
| Tamil | English | ????? ??? ??????? ????. | Tamil is an ancient language. |
| Telugu | English | ?????? ???????????? ?????? ?????? ????. | Telugu is one of the major languages of India. |

## Model Performance Note

The default model in this project is `facebook/nllb-200-distilled-600M`, which gives better translation quality but needs more RAM and works best with a GPU.

For faster testing on a normal laptop, change the model name to:

```python
model_name = "facebook/nllb-200-distilled-600M"
```

This project uses the distilled model, greedy decoding, short input limits, and a shorter output limit for quicker development, testing, and classroom demonstrations.

## Streamlit App Screenshot

The following preview shows the planned Streamlit app interface with Translate, Architecture, and History tabs.

![Streamlit app preview](streamlit_app_preview.svg)

After running the Streamlit app on your machine, you can replace or add more screenshots from the actual browser output.

## 12. Streamlit Web Application Code

Run this cell to create `app.py`, then start the app with `streamlit run app.py`.

In [ ]:
%%writefile app.py
from __future__ import annotations

import sqlite3
from time import perf_counter
from dataclasses import dataclass, field
from datetime import datetime
from html import escape
from io import BytesIO
from pathlib import Path

import streamlit as st

LANGUAGE_OPTIONS = {
    "English": "eng_Latn",
    "Assamese": "asm_Beng",
    "Bengali": "ben_Beng",
    "Bodo": "brx_Deva",
    "Dogri": "doi_Deva",
    "Gujarati": "guj_Gujr",
    "Hindi": "hin_Deva",
    "Kannada": "kan_Knda",
    "Kashmiri": "kas_Arab",
    "Konkani": "gom_Deva",
    "Maithili": "mai_Deva",
    "Malayalam": "mal_Mlym",
    "Manipuri": "mni_Beng",
    "Marathi": "mar_Deva",
    "Nepali": "npi_Deva",
    "Odia": "ory_Orya",
    "Punjabi": "pan_Guru",
    "Sanskrit": "san_Deva",
    "Santali": "sat_Olck",
    "Sindhi": "snd_Arab",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Urdu": "urd_Arab",
}

DETECTION_CODES = {
    "en": "English",
    "as": "Assamese",
    "bn": "Bengali",
    "gu": "Gujarati",
    "hi": "Hindi",
    "kn": "Kannada",
    "ml": "Malayalam",
    "mr": "Marathi",
    "ne": "Nepali",
    "or": "Odia",
    "pa": "Punjabi",
    "ta": "Tamil",
    "te": "Telugu",
    "ur": "Urdu",
}
TTS_CODES = {
    "English": "en",
    "Assamese": "as",
    "Bengali": "bn",
    "Gujarati": "gu",
    "Hindi": "hi",
    "Kannada": "kn",
    "Malayalam": "ml",
    "Marathi": "mr",
    "Nepali": "ne",
    "Punjabi": "pa",
    "Tamil": "ta",
    "Telugu": "te",
    "Urdu": "ur",
}

ARCHITECTURE_DOT = 'digraph translation_architecture {\n    graph [rankdir=TB, bgcolor="transparent", pad="0.3", nodesep="0.45", ranksep="0.55"];\n    node [shape=box, style="rounded,filled", color="#506070", fillcolor="#F8FAFC", fontname="Arial", fontsize=10];\n    edge [color="#64748B", arrowsize=0.8];\n\n    user [label="User"];\n    app [label="Streamlit Web Application", fillcolor="#E0F2FE"];\n    text_input [label="Text Input"];\n    voice_input [label="Voice Input"];\n    stt [label="Speech-to-Text"];\n    source_text [label="Source Text"];\n    language [label="Manual Language Selection / Auto Detection"];\n    tokenizer [label="Tokenizer"];\n    encoder [label="Transformer Encoder"];\n    attention_scores [label="Attention Scores"];\n    attention_softmax [label="Softmax"];\n    attention_weights [label="Attention Weights"];\n    decoder [label="Transformer Decoder"];\n    cross_attention [label="Cross-Attention"];\n    output_scores [label="Output Token Scores"];\n    output_softmax [label="Softmax"];\n    probabilities [label="Token Probabilities"];\n    translated [label="Translated Text", fillcolor="#DCFCE7"];\n    tts [label="Text-to-Speech"];\n    audio [label="Voice Output"];\n    db [label="SQLite Translation History"];\n    history [label="History View"];\n\n    user -> app;\n    app -> text_input;\n    app -> voice_input;\n    voice_input -> stt -> source_text;\n    text_input -> source_text;\n    source_text -> language -> tokenizer -> encoder;\n    encoder -> attention_scores -> attention_softmax -> attention_weights -> decoder;\n    decoder -> cross_attention -> output_scores -> output_softmax -> probabilities -> translated;\n    translated -> tts -> audio;\n    translated -> db -> history;\n    translated -> app;\n    audio -> app;\n    history -> app;\n}'

def get_language_code(language_name: str) -> str:
    return LANGUAGE_OPTIONS[language_name]

def get_tts_code(language_name: str) -> str:
    return TTS_CODES[language_name]

def detect_supported_language(text: str) -> str:
    from langdetect import detect
    detected_code = detect(text)
    if detected_code not in DETECTION_CODES:
        raise ValueError("Automatic detection is best-effort. Use manual selection if detection fails.")
    return DETECTION_CODES[detected_code]

class TranslationError(RuntimeError):
    pass

@dataclass(slots=True)
class TransformerTranslator:
    model_name: str = "facebook/nllb-200-distilled-600M"
    max_length: int = 64
    num_beams: int = 1
    _tokenizer: object | None = field(default=None, init=False, repr=False)
    _model: object | None = field(default=None, init=False, repr=False)

    def _load_model(self):
        if self._tokenizer is not None and self._model is not None:
            return self._tokenizer, self._model
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        try:
            self._tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self._model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
            self._model.eval()
        except Exception as exc:
            raise TranslationError("Model could not be loaded. Check internet access or local model cache.") from exc
        return self._tokenizer, self._model

    def detect_language(self, text: str) -> str:
        try:
            return detect_supported_language(text)
        except Exception as exc:
            raise TranslationError("Automatic language detection failed. Select the source language manually.") from exc

    def split_into_sentences(self, text: str):
        import re
        cleaned_text = " ".join(text.split())
        return [part.strip() for part in re.split(r"(?<=[.!??])\s+", cleaned_text) if part.strip()]

    def quality_warnings(self, text: str, source_mode: str):
        warnings = []
        if len(text.split()) < 3:
            warnings.append("Input is very short, so translation may be less reliable.")
        if source_mode == "Automatic detection":
            warnings.append("Manual source language selection gives better accuracy.")
        if len(text) > 700:
            warnings.append("Long text will be split into sentences for better translation quality.")
        return warnings

    def translate_sentence(self, text: str, source_language: str, target_language: str) -> str:
        tokenizer, model = self._load_model()
        tokenizer.src_lang = get_language_code(source_language)
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=96)
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(get_language_code(target_language))
        import torch
        torch.set_num_threads(2)

        with torch.inference_mode():
            output_tokens = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=self.max_length,
                num_beams=self.num_beams,
            )
        return tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0]

    def translate(self, text: str, source_language: str, target_language: str) -> str:
        if not text.strip():
            raise TranslationError("Please enter text before translating.")
        if len(text.split()) > 25:
            raise TranslationError("For under-5-second demo translation, enter 25 words or fewer.")
        if source_language == target_language:
            raise TranslationError("Source and target languages must be different.")
        # Fast demo mode: translate the input in one model call.
        return self.translate_sentence(text, source_language, target_language)

class TranslationHistory:
    def __init__(self, database_path: str | Path = "translation_history.db"):
        self.database_path = Path(database_path)
        self._create_table()

    def _create_table(self):
        with sqlite3.connect(self.database_path) as connection:
            connection.execute("""
                CREATE TABLE IF NOT EXISTS translation_history (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_language TEXT NOT NULL,
                    target_language TEXT NOT NULL,
                    source_text TEXT NOT NULL,
                    translated_text TEXT NOT NULL,
                    created_at TEXT NOT NULL
                )
            """)

    def add(self, source_language: str, target_language: str, source_text: str, translated_text: str):
        with sqlite3.connect(self.database_path) as connection:
            connection.execute(
                """
                INSERT INTO translation_history
                (source_language, target_language, source_text, translated_text, created_at)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    source_language,
                    target_language,
                    source_text,
                    translated_text,
                    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                ),
            )

    def recent(self, limit: int = 20):
        with sqlite3.connect(self.database_path) as connection:
            return connection.execute(
                """
                SELECT source_language, target_language, source_text, translated_text, created_at
                FROM translation_history
                ORDER BY id DESC
                LIMIT ?
                """,
                (limit,),
            ).fetchall()

    def clear(self):
        with sqlite3.connect(self.database_path) as connection:
            connection.execute("DELETE FROM translation_history")

def speech_to_text(audio_file, language_name: str) -> str:
    import speech_recognition as sr
    recognizer = sr.Recognizer()
    recognizer.energy_threshold = 300
    recognizer.dynamic_energy_threshold = True
    with sr.AudioFile(audio_file) as source:
        recognizer.adjust_for_ambient_noise(source, duration=0.5)
        audio = recognizer.record(source)
    return recognizer.recognize_google(audio, language=get_tts_code(language_name))

def text_to_speech_bytes(text: str, language_name: str) -> bytes:
    from gtts import gTTS
    buffer = BytesIO()
    gTTS(text=text, lang=get_tts_code(language_name)).write_to_fp(buffer)
    return buffer.getvalue()

@st.cache_resource(show_spinner="Loading Transformer model...")
def load_translator():
    return TransformerTranslator()

@st.cache_resource
def load_history():
    return TranslationHistory()

def apply_theme():
    st.markdown(
        """
        <style>
        :root {
            --surface: #ffffff;
            --surface-soft: #f8fafc;
            --line: #d8e0ea;
            --ink: #0f172a;
            --muted: #5f6f84;
            --brand: #0f766e;
            --brand-dark: #115e59;
            --accent: #2563eb;
            --success-bg: #ecfdf5;
            --success-line: #a7f3d0;
            --warn-bg: #fff7ed;
            --warn-line: #fed7aa;
        }
        .main .block-container {
            max-width: 1220px;
            padding-top: 2rem;
            padding-bottom: 3rem;
        }
        h1, h2, h3 {
            letter-spacing: 0;
        }
        .app-hero {
            border: 1px solid var(--line);
            background: linear-gradient(135deg, #f8fafc 0%, #ecfeff 48%, #eef2ff 100%);
            padding: 28px 30px;
            border-radius: 8px;
            margin-bottom: 20px;
        }
        .app-title {
            color: var(--ink);
            font-size: 34px;
            font-weight: 800;
            margin: 0;
        }
        .app-subtitle {
            color: var(--muted);
            font-size: 16px;
            margin-top: 8px;
            max-width: 860px;
            line-height: 1.55;
        }
        .metric-row {
            display: grid;
            grid-template-columns: repeat(4, minmax(0, 1fr));
            gap: 12px;
            margin: 16px 0 8px;
        }
        .metric-card {
            background: var(--surface);
            border: 1px solid var(--line);
            border-radius: 8px;
            padding: 14px 16px;
        }
        .metric-label {
            color: var(--muted);
            font-size: 12px;
            text-transform: uppercase;
            letter-spacing: .06em;
            font-weight: 750;
        }
        .metric-value {
            color: var(--ink);
            font-size: 20px;
            font-weight: 800;
            margin-top: 4px;
        }
        .section-panel {
            background: var(--surface);
            border: 1px solid var(--line);
            border-radius: 8px;
            padding: 18px;
            margin-bottom: 16px;
        }
        .panel-title {
            color: var(--ink);
            font-size: 18px;
            font-weight: 800;
            margin-bottom: 6px;
        }
        .panel-help {
            color: var(--muted);
            font-size: 13px;
            margin-bottom: 14px;
        }
        .output-box {
            background: var(--success-bg);
            border: 1px solid var(--success-line);
            border-radius: 8px;
            padding: 18px;
            margin-top: 16px;
        }
        .output-label {
            color: #166534;
            font-size: 13px;
            font-weight: 800;
            text-transform: uppercase;
            letter-spacing: .06em;
            margin-bottom: 8px;
        }
        .output-text {
            color: #052e16;
            font-size: 21px;
            line-height: 1.65;
            font-weight: 650;
            word-break: break-word;
        }
        .status-pill {
            display: inline-block;
            background: #e0f2fe;
            color: #075985;
            border: 1px solid #bae6fd;
            border-radius: 999px;
            padding: 6px 10px;
            font-size: 13px;
            font-weight: 750;
            margin-top: 10px;
        }
        .stButton > button {
            border-radius: 7px;
            font-weight: 750;
        }
        div[data-testid="stTabs"] button {
            font-weight: 750;
        }
        @media (max-width: 760px) {
            .metric-row {
                grid-template-columns: repeat(2, minmax(0, 1fr));
            }
            .app-title {
                font-size: 27px;
            }
        }
        </style>
        """,
        unsafe_allow_html=True,
    )


def render_metric(label: str, value: str):
    st.markdown(
        f"""
        <div class="metric-card">
            <div class="metric-label">{label}</div>
            <div class="metric-value">{value}</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


st.set_page_config(page_title="Multilingual Translator", layout="wide")
apply_theme()
st.markdown(
    """
    <div class="app-hero">
        <div class="app-title">Multilingual Language Translation System</div>
        <div class="app-subtitle">
            A Transformer-based translation workspace for Indian languages with text translation,
            speech input, voice output, architecture visualization, and SQLite translation history.
        </div>
    </div>
    """,
    unsafe_allow_html=True,
)

metric_cols = st.columns(4)
with metric_cols[0]:
    render_metric("Languages", "22 + English")
with metric_cols[1]:
    render_metric("Model", "NLLB Distilled")
with metric_cols[2]:
    render_metric("Mode", "Fast Demo")
with metric_cols[3]:
    render_metric("Storage", "SQLite")

history = load_history()
language_names = list(LANGUAGE_OPTIONS.keys())
translate_tab, architecture_tab, history_tab = st.tabs(["Translate", "Architecture", "History"])

with translate_tab:
    st.markdown('<div class="section-panel">', unsafe_allow_html=True)
    st.markdown('<div class="panel-title">Translation Workspace</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="panel-help">Fast demo mode is optimized for short inputs. Use 25 words or fewer after the model has loaded.</div>',
        unsafe_allow_html=True,
    )

    mode_col, source_col, target_col = st.columns([1.2, 1, 1])
    with mode_col:
        source_mode = st.radio(
            "Source mode",
            ["Manual selection", "Automatic detection"],
            horizontal=True,
        )
    with source_col:
        source_language = st.selectbox("Source language", language_names)
    with target_col:
        target_language = st.selectbox("Target language", language_names, index=1)

    input_col, tools_col = st.columns([1.45, 1])
    with input_col:
        text = st.text_area(
            "Text to translate",
            height=190,
            placeholder="Enter a short sentence for faster translation...",
        )
        word_count = len(text.split())
        st.caption(f"Word count: {word_count}/25")
        translate_clicked = st.button("Translate Text", type="primary", use_container_width=True)

    with tools_col:
        st.markdown('<div class="panel-title">Voice Input</div>', unsafe_allow_html=True)
        speech_language = st.selectbox("Speech language", language_names, key="speech_language")
        st.caption("Use clear WAV, AIFF, or FLAC audio with low background noise.")
        audio_file = st.file_uploader(
            "Upload audio",
            type=["wav", "aiff", "aif", "flac"],
        )
        if st.button("Convert Voice To Text", use_container_width=True) and audio_file is not None:
            try:
                st.session_state.voice_text = speech_to_text(audio_file, speech_language)
                st.success("Voice converted to text.")
            except Exception as exc:
                st.error(f"Speech recognition failed: {exc}")

        if "voice_text" in st.session_state:
            st.text_area("Recognized text", st.session_state.voice_text, height=92)
            if st.button("Use Recognized Text", use_container_width=True):
                text = st.session_state.voice_text

    if translate_clicked:
        try:
            translator = load_translator()
            for warning in translator.quality_warnings(text, source_mode):
                st.warning(warning)
            actual_source_language = translator.detect_language(text) if source_mode == "Automatic detection" else source_language
            started_at = perf_counter()
            translated_text = translator.translate(text, actual_source_language, target_language)
            elapsed = perf_counter() - started_at
            history.add(actual_source_language, target_language, text, translated_text)
            st.session_state.latest_translation = translated_text
            st.session_state.latest_target_language = target_language
            st.session_state.latest_source_language = actual_source_language
            st.session_state.latest_elapsed = elapsed
        except Exception as exc:
            st.error(f"Translation failed: {exc}")

    if "latest_translation" in st.session_state:
        safe_translation = escape(st.session_state.latest_translation)
        safe_source = escape(st.session_state.latest_source_language)
        safe_target = escape(st.session_state.latest_target_language)
        st.markdown(
            f"""
            <div class="output-box">
                <div class="output-label">Translated Text</div>
                <div class="output-text">{safe_translation}</div>
                <div class="status-pill">
                    {safe_source} to {safe_target}
                    | {st.session_state.latest_elapsed:.2f}s after model load
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )
        voice_col, copy_col = st.columns([1, 1])
        with voice_col:
            if st.button("Generate Voice Output", use_container_width=True):
                try:
                    audio_bytes = text_to_speech_bytes(
                        st.session_state.latest_translation,
                        st.session_state.latest_target_language,
                    )
                    st.audio(audio_bytes, format="audio/mp3")
                except Exception as exc:
                    st.error(f"Text-to-speech failed: {exc}")
        with copy_col:
            st.download_button(
                "Download Translation",
                data=st.session_state.latest_translation,
                file_name="translation.txt",
                mime="text/plain",
                use_container_width=True,
            )

    st.markdown("</div>", unsafe_allow_html=True)

with architecture_tab:
    st.markdown('<div class="section-panel">', unsafe_allow_html=True)
    st.markdown('<div class="panel-title">Project Architecture</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="panel-help">The model uses encoder-decoder Transformer layers with attention, cross-attention, and Softmax.</div>',
        unsafe_allow_html=True,
    )
    st.graphviz_chart(ARCHITECTURE_DOT, use_container_width=True)
    st.info("Softmax is used inside attention to create attention weights and at the decoder output to create token probabilities.")
    st.markdown("</div>", unsafe_allow_html=True)

with history_tab:
    st.markdown('<div class="section-panel">', unsafe_allow_html=True)
    st.markdown('<div class="panel-title">Translation History</div>', unsafe_allow_html=True)
    records = history.recent()
    if not records:
        st.info("No translation history yet.")
    for source_language, target_language, source_text, translated_text, created_at in records:
        with st.expander(f"{source_language} to {target_language} - {created_at}"):
            left, right = st.columns(2)
            with left:
                st.caption("Input")
                st.write(source_text)
            with right:
                st.caption("Output")
                st.write(translated_text)
    if records and st.button("Clear history"):
        history.clear()
        st.rerun()
    st.markdown("</div>", unsafe_allow_html=True)


## Conclusion

The proposed system successfully demonstrates a Transformer-based multilingual translation workflow for Indian languages. It combines modern neural machine translation with a user-friendly Streamlit interface, speech input, speech output, automatic language detection, translation history, and an architecture view explaining encoder, decoder, attention, and Softmax. While no machine translation model can guarantee perfect results for every sentence, the use of a high-quality NLLB model, beam search, sentence splitting, and manual source language selection improves translation reliability. The system can be further enhanced by fine-tuning on verified Indian language parallel datasets.

## 13. Run The Streamlit App

After creating `app.py`, run this command in a terminal or notebook cell.

In [ ]:
# !streamlit run app.py

## Final How To Run

Run the notebook cells in order. Then create and launch the Streamlit app using the app-generation cell.

Commands:

```bash
pip install streamlit torch transformers sentencepiece sacremoses langdetect SpeechRecognition gTTS sacrebleu
streamlit run app.py
```

If the large model is slow or your system does not have enough memory, switch from:

```python
facebook/nllb-200-distilled-600M
```

to:

```python
facebook/nllb-200-distilled-600M
```

The first run needs internet access to download the model. Voice input and voice output also require internet access.